# ViSoBERT -- Baseline (no augmentation)

Train trên subset `baseline_segmented` (đã phân từ sẵn trên Hugging Face Hub), đánh giá trên `tdtu_dev` / `tdtu_test` (đã đóng băng). Đây là điểm neo (anchor) để so sánh với 4 thí nghiệm augmentation.

**Khác biệt so với bản PhoBERT:**
- Model: `uitnlp/visobert` (XLM-R based, train trên Vietnamese social media text).
- Tokenizer: dùng **fast** tokenizer (`use_fast=True`).
- **Không** áp dụng word segmentation: ViSoBERT huấn luyện trên raw text nên sẽ **bỏ dấu `_`** của VnCoreNLP khỏi cột `text` để quay về raw gần đúng.
- Bỏ dependency `py_vncorenlp`.

**Chạy trên Google Colab** -- cần mount Drive và có sẵn secret `HF_TOKEN`.

## Dependencies

In [1]:
!pip install -q transformers datasets huggingface_hub scikit-learn accelerate sentencepiece
!pip install -q -U datasets

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Enable GPU in Runtime > Change runtime type.")

Device: cuda
GPU: Tesla T4


In [3]:
from huggingface_hub import notebook_login, whoami

# Hiển thị ô nhập Token
notebook_login()

# Kiểm tra tài khoản sau khi nhập thành công
user_info = whoami()
print(f"Đã đăng nhập thành công với tài khoản: {user_info['name']}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Đã đăng nhập thành công với tài khoản: AnoraLee


In [4]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [5]:
MODEL_NAME = "uitnlp/visobert"
MAX_LENGTH = 128
SEED = 42
LABELS = ["CLEAN", "OFFENSIVE", "HATE"]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

# Repo data trên Hugging Face Hub
HF_DATA_REPO = "AnoraLee/tdtu_vietnamese_hsd_final"
SUBSET_NAME = "baseline_segmented"

## Load dữ liệu từ Hugging Face Hub + chuẩn hoá cho ViSoBERT

Data trên Hub đã được làm sạch (chuẩn hoá teencode + gộp nguồn + loại conflict/leakage). Vì ViSoBERT không dùng phân từ, ta **bỏ dấu `_`** trong cột `text` (nếu có) để quay về dạng raw gần đúng. Nếu Hub có sẵn cột `text_raw`, sẽ ưu tiên dùng nó.

In [6]:
from datasets import load_dataset
import re

def to_visobert_text(df):
    """Chuẩn hoá cột text cho ViSoBERT:
    - Nếu có cột 'text_raw' -> dùng trực tiếp.
    - Nếu chỉ có 'text' (đã phân từ) -> bỏ dấu '_' để quay về raw.
    """
    df = df.copy()
    if "text_raw" in df.columns:
        df["text"] = df["text_raw"].astype(str)
    else:
        # Bỏ dấu '_' giữa các âm tiết: 'học_sinh' -> 'học sinh'
        df["text"] = df["text"].astype(str).str.replace("_", " ", regex=False)
        df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()
    return df

print(f"Đang tải dữ liệu [{SUBSET_NAME}] từ Hugging Face Hub...")
dataset = load_dataset(HF_DATA_REPO, SUBSET_NAME)
print(dataset)

train_df = dataset["tdtu_train"].to_pandas()
dev_df   = dataset["tdtu_dev"].to_pandas()
test_df  = dataset["tdtu_test"].to_pandas()

train_df = train_df.dropna(subset=["text", "label"]).reset_index(drop=True)
dev_df   = dev_df.dropna(subset=["text", "label"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["text", "label"]).reset_index(drop=True)

train_df = to_visobert_text(train_df)
dev_df   = to_visobert_text(dev_df)
test_df  = to_visobert_text(test_df)

print(f"Train:      {train_df.shape}")
print(f"Validation: {dev_df.shape}")
print(f"Test:       {test_df.shape}")
print("Label distribution (train):", train_df["label"].value_counts().to_dict())
print("Sample text:", train_df["text"].iloc[0][:200])

Đang tải dữ liệu [baseline_segmented] từ Hugging Face Hub...


README.md:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

baseline_segmented/tdtu_train-00000-of-0(…): reconstructing file:   0%|          |  0.00B / 8.69MB            

baseline_segmented/tdtu_train-00000-of-0(…): downloading bytes:           |  0.00B            

baseline_segmented/tdtu_dev-00000-of-000(…): reconstructing file:   0%|          |  0.00B /  127kB            

...

baseline_segmented/tdtu_test-00000-of-00(…): downloading bytes:           |  0.00B            

Generating tdtu_train split:   0%|          | 0/86719 [00:00<?, ? examples/s]

Generating tdtu_dev split:   0%|          | 0/2650 [00:00<?, ? examples/s]

Generating tdtu_test split:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    tdtu_train: Dataset({
        features: ['text', 'label'],
        num_rows: 86719
    })
    tdtu_dev: Dataset({
        features: ['text', 'label'],
        num_rows: 2650
    })
    tdtu_test: Dataset({
        features: ['text', 'label'],
        num_rows: 6576
    })
})
Train:      (86716, 2)
Validation: (2650, 2)
Test:       (6576, 2)
Label distribution (train): {'CLEAN': 61408, 'OFFENSIVE': 19576, 'HATE': 5732}
Sample text: Khó vẽ nụ cười


## Tokenizer + hàm dùng chung

In [7]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def prepare_split(df):
    output = df[["text", "label"]].dropna().copy()
    assert output["label"].isin(LABELS).all(), "Unexpected label found."
    output["label"] = output["label"].map(label2id)
    return Dataset.from_pandas(output, preserve_index=False)

def build_tokenized_dataset(train_df, dev_df, test_df):
    dataset = DatasetDict({
        "train": prepare_split(train_df),
        "validation": prepare_split(dev_df),
        "test": prepare_split(test_df),
    })
    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    model_columns = [
        c for c in ["input_ids", "attention_mask", "labels"]
        if c in tokenized["train"].column_names
    ]
    tokenized.set_format("torch", columns=model_columns)
    print(tokenized)
    return tokenized

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  471kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

In [8]:
tokenized = build_tokenized_dataset(train_df, dev_df, test_df)

Map:   0%|          | 0/86716 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 86716
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


## Metrics

In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Unified metric schema -- MUST stay identical across all notebooks
    so results in the report are directly comparable."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "hate_f1": f1_score(labels, predictions, labels=[LABELS.index("HATE")], average="macro"),
    }

## Train

In [10]:
import json
from transformers import TrainingArguments, AutoModelForSequenceClassification, Trainer, set_seed, EarlyStoppingCallback

set_seed(SEED)

def train_experiment(experiment_name, model_dir_name, tokenized, push_to_hub_repo=None):
    """Load a FRESH ViSoBERT model and train one experiment end to end.
    Always instantiates a new model -- never reuses a model/trainer object
    from a previous cell, to avoid accidental weight leakage between runs."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, label2id=label2id, id2label=id2label,
    )
    model_dir = MODELS_DIR / model_dir_name

    training_args = TrainingArguments(
        output_dir=str(model_dir / "_checkpoints"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print(f"[{experiment_name}] test metrics: {test_metrics}")

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    # Lưu metrics vào results/metrics/metrics_<experiment>.json (chung 1 thư mục)
    metrics_dir = RESULTS_DIR / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = metrics_dir / f"metrics_{experiment_name}.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, indent=2, ensure_ascii=False)

    print(f"Model saved to:   {model_dir}")
    print(f"Metrics saved to: {metrics_path}")

    if push_to_hub_repo:
        api = HfApi()
        api.create_repo(repo_id=push_to_hub_repo, repo_type="model", private=True, exist_ok=True)
        trainer.model.push_to_hub(push_to_hub_repo, private=True,
                                  commit_message=f"Upload ViSoBERT {experiment_name} model")
        tokenizer.push_to_hub(push_to_hub_repo, private=True,
                              commit_message="Upload ViSoBERT tokenizer")
        api.upload_file(
            path_or_fileobj=str(metrics_path),
            path_in_repo="test_metrics.json",
            repo_id=push_to_hub_repo, repo_type="model",
        )
        print(f"Uploaded: https://huggingface.co/{push_to_hub_repo}")

    return trainer, test_metrics

In [11]:
trainer, test_metrics = train_experiment(
    experiment_name="visobert_baseline",
    model_dir_name="baseline_visobert",
    tokenized=tokenized,
    push_to_hub_repo=f"{whoami()['name']}/vietnamese-hsd-visobert-baseline",
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  390MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  390MB            

...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Hate F1
0.177322,0.448180,4,0.845043,0.615430,0.850586,0.555024


[visobert_baseline] test metrics: {'test_loss': 0.4481799006462097, 'test_accuracy': 0.8450425790754258, 'test_macro_f1': 0.6154298523418796, 'test_weighted_f1': 0.8505857848898963, 'test_hate_f1': 0.5550239234449761}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:   /content/drive/MyDrive/Hate_Speech_Detection/models/baseline_visobert
Metrics saved to: /content/drive/MyDrive/Hate_Speech_Detection/results/metrics/metrics_visobert_baseline.json


NameError: name 'HfApi' is not defined

In [12]:
from huggingface_hub import HfApi, whoami
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Định nghĩa lại đường dẫn
experiment_name = "visobert_baseline"
push_to_hub_repo = f"{whoami()['name']}/vietnamese-hsd-visobert-baseline"
model_dir = MODELS_DIR / "baseline_visobert"

# Sửa lại đường dẫn file json theo đúng setup bạn đã làm
metrics_path = RESULTS_DIR / "metrics" / f"metrics_{experiment_name}.json"

print(f"Đang đẩy model từ {model_dir} lên {push_to_hub_repo}...")

# 1. Load lại model và tokenizer đã train xong từ Drive
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# 2. Đẩy lên Hugging Face
api = HfApi()
api.create_repo(repo_id=push_to_hub_repo, repo_type="model", private=False, exist_ok=True)

model.push_to_hub(push_to_hub_repo, private=False, commit_message=f"Upload ViSoBERT {experiment_name} model")
tokenizer.push_to_hub(push_to_hub_repo, private=False, commit_message="Upload ViSoBERT tokenizer")

# 3. Đẩy kèm file json metrics
if metrics_path.exists():
    api.upload_file(
        path_or_fileobj=str(metrics_path),
        path_in_repo="test_metrics.json",
        repo_id=push_to_hub_repo, repo_type="model",
    )

print(f"✅ Hoàn tất! Link mô hình của bạn: https://huggingface.co/{push_to_hub_repo}")

Đang đẩy model từ /content/drive/MyDrive/Hate_Speech_Detection/models/baseline_visobert lên AnoraLee/vietnamese-hsd-visobert-baseline...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b8p6vjv/model.safetensors:   0%|          |  554kB /  390MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

✅ Hoàn tất! Link mô hình của bạn: https://huggingface.co/AnoraLee/vietnamese-hsd-visobert-baseline
